In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score

from src.ensemble import ScoreCombiner

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "evaluation_results"
MODEL_DIR = Path.cwd() / "trained_models"

print("Project root:", PROJECT_ROOT)

Project root: /home/kalpe/projects/adaptive_rl_anomaly_detection


In [2]:
PROCESSED_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "datasets"
    / "processed"
    / "cicids2017_processed.parquet"
)

df = pd.read_parquet(PROCESSED_PATH)

print("Dataset shape:", df.shape)
print(df.columns[-5:])

Dataset shape: (2520798, 71)
Index(['Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'Label'], dtype='str')


In [3]:
X = df.drop(columns=["Label"])
y = (df["Label"] != 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_dev, X_val, y_dev, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.10,
    random_state=42,
    stratify=y_train,
)

X_dev = X_dev.reset_index(drop=True)
y_dev = y_dev.reset_index(drop=True)

X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("Development:", X_dev.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

Development: (1814974, 70)
Validation : (201664, 70)
Test       : (504160, 70)


In [4]:
from src.models.isolation_forest import IsolationForestModel
from src.models.local_outlier_factor import LocalOutlierFactorModel
from src.models.one_class_svm import OneClassSVMModel
from src.models.autoencoder import AutoEncoderModel

if_model = IsolationForestModel.load(
    MODEL_DIR / "isolation_forest.joblib"
)

lof_model = LocalOutlierFactorModel.load(
    MODEL_DIR / "local_outlier_factor.joblib"
)

svm_model = OneClassSVMModel.load(
    MODEL_DIR / "one_class_svm.joblib"
)

ae_model = AutoEncoderModel.load(
    MODEL_DIR / "autoencoder.joblib"
)

print("✓ Four models loaded")

2026-08-12 14:15:20 | INFO     | src.models.isolation_forest | Isolation Forest loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/isolation_forest.joblib
2026-08-12 14:15:20 | INFO     | src.models.local_outlier_factor | Local Outlier Factor loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/local_outlier_factor.joblib
2026-08-12 14:15:20 | INFO     | src.models.one_class_svm | One-Class SVM loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/one_class_svm.joblib
2026-08-12 14:15:21 | INFO     | src.models.autoencoder | AutoEncoder loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/autoencoder.joblib


✓ Four models loaded


In [5]:
validation_scores_raw = {
    "if": np.asarray(if_model.anomaly_score(X_val)),
    "lof": np.asarray(lof_model.anomaly_score(X_val)),
    "svm": np.asarray(svm_model.anomaly_score(X_val)),
    "ae": np.asarray(ae_model.anomaly_score(X_val)),
}

for name, scores in validation_scores_raw.items():
    print(
        f"{name}: "
        f"shape={scores.shape}, "
        f"min={scores.min():.6f}, "
        f"max={scores.max():.6f}"
    )

if: shape=(201664,), min=0.319992, max=0.744344
lof: shape=(201664,), min=-2.296478, max=8985.527440
svm: shape=(201664,), min=-73.646861, max=100.205412
ae: shape=(201664,), min=0.000068, max=2774.155029


In [6]:
score_combiner = ScoreCombiner(
    weights={
        "if": 0.25,
        "lof": 0.25,
        "svm": 0.25,
        "ae": 0.25,
    },
    normalization="percentile",
)

score_combiner.fit(validation_scores_raw)

normalized_scores = (
    score_combiner.get_normalized_scores(
        validation_scores_raw
    )
)

print("✓ Scores normalized")

✓ Scores normalized


In [7]:
state_s1 = np.column_stack([
    normalized_scores["if"],
    normalized_scores["lof"],
    normalized_scores["svm"],
    normalized_scores["ae"],
])

s1_columns = [
    "if_score",
    "lof_score",
    "svm_score",
    "ae_score",
]

print("S1 shape:", state_s1.shape)

S1 shape: (201664, 4)


In [8]:
score_matrix = state_s1

disagreement_stats = np.column_stack([
    score_matrix.mean(axis=1),
    score_matrix.std(axis=1),
    score_matrix.min(axis=1),
    score_matrix.max(axis=1),
    score_matrix.max(axis=1) - score_matrix.min(axis=1),
])

disagreement_columns = [
    "score_mean",
    "score_std",
    "score_min",
    "score_max",
    "score_range",
]

print(
    "Disagreement statistics:",
    disagreement_stats.shape
)

Disagreement statistics: (201664, 5)


In [9]:
state_s2 = np.column_stack([
    state_s1,
    disagreement_stats,
])

s2_columns = s1_columns + disagreement_columns

print("S2 shape:", state_s2.shape)

S2 shape: (201664, 9)


In [10]:
mi_scores = mutual_info_classif(
    X_dev,
    y_dev,
    random_state=42,
)

feature_ranking = (
    pd.DataFrame({
        "feature": X_dev.columns,
        "mutual_information": mi_scores,
    })
    .sort_values(
        "mutual_information",
        ascending=False,
    )
    .reset_index(drop=True)
)

feature_ranking.head(15)

,feature,mutual_information
0,Average Packet Size,0.354807
1,Packet Length Variance,0.340347
2,Packet Length Std,0.340285
3,Packet Length Mean,0.328024
4,Avg Bwd Segment Size,0.307678
5,Bwd Packet Length Mean,0.307429
6,Subflow Bwd Bytes,0.299941
7,Total Length of Bwd Packets,0.299694
8,Init_Win_bytes_backward,0.295274
9,Destination Port,0.293482


In [11]:
selected_features = (
    feature_ranking
    .head(5)["feature"]
    .tolist()
)

print("Selected traffic features:")

for i, feature in enumerate(
    selected_features,
    start=1,
):
    print(f"{i}. {feature}")

Selected traffic features:
1. Average Packet Size
2. Packet Length Variance
3. Packet Length Std
4. Packet Length Mean
5. Avg Bwd Segment Size


In [12]:
traffic_scaler = StandardScaler()

traffic_context = traffic_scaler.fit_transform(
    X_val[selected_features]
)

print(
    "Traffic context shape:",
    traffic_context.shape
)

Traffic context shape: (201664, 5)


In [13]:
state_s3 = np.column_stack([
    state_s2,
    traffic_context,
])

s3_columns = (
    s2_columns
    + selected_features
)

print("S3 shape:", state_s3.shape)

S3 shape: (201664, 14)


In [14]:
states = {
    "S1_scores_only": state_s1,
    "S2_scores_disagreement": state_s2,
    "S3_scores_disagreement_traffic": state_s3,
}

for name, state in states.items():
    print(
        f"{name}: "
        f"samples={state.shape[0]}, "
        f"features={state.shape[1]}"
    )

S1_scores_only: samples=201664, features=4
S2_scores_disagreement: samples=201664, features=9
S3_scores_disagreement_traffic: samples=201664, features=14


In [15]:
for state_name, state in states.items():

    print("=" * 70)
    print(state_name)

    state_df = pd.DataFrame(state)

    auc_values = []

    for i in range(state.shape[1]):
        auc = roc_auc_score(
            y_val,
            state[:, i],
        )

        auc_values.append(
            max(auc, 1.0 - auc)
        )

    print(
        "Mean feature discrimination:",
        np.mean(auc_values)
    )

    print(
        "Best feature discrimination:",
        np.max(auc_values)
    )

S1_scores_only
Mean feature discrimination: 0.7279533444841717
Best feature discrimination: 0.8999442524651569
S2_scores_disagreement
Mean feature discrimination: 0.6804500736798466
Best feature discrimination: 0.8999442524651569
S3_scores_disagreement_traffic
Mean feature discrimination: 0.6725269130449331
Best feature discrimination: 0.8999442524651569


In [16]:
STATE_DIR = PROJECT_ROOT / "evaluation_results" / "rl_states"
STATE_DIR.mkdir(parents=True, exist_ok=True)

np.save(
    STATE_DIR / "state_s1.npy",
    state_s1,
)

np.save(
    STATE_DIR / "state_s2.npy",
    state_s2,
)

np.save(
    STATE_DIR / "state_s3.npy",
    state_s3,
)

joblib.dump(
    {
        "S1": s1_columns,
        "S2": s2_columns,
        "S3": s3_columns,
        "selected_traffic_features": selected_features,
    },
    STATE_DIR / "state_metadata.joblib",
)

print("✓ S1, S2 and S3 saved")


✓ S1, S2 and S3 saved


In [17]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
)

state_results = []

for state_name, state in states.items():

    clf = ExtraTreesClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    )

    clf.fit(state, y_val)

    probabilities = clf.predict_proba(state)[:, 1]
    predictions = (probabilities >= 0.5).astype(np.int8)

    state_results.append({
        "state": state_name,
        "dimensions": state.shape[1],
        "roc_auc": roc_auc_score(y_val, probabilities),
        "pr_auc": average_precision_score(y_val, probabilities),
        "f1": f1_score(y_val, predictions),
    })

state_comparison = pd.DataFrame(state_results)

state_comparison.sort_values(
    "pr_auc",
    ascending=False,
)

,state,dimensions,roc_auc,pr_auc,f1
2,S3_scores_disagreement_traffic,14,0.992354,0.969068,0.879346
0,S1_scores_only,4,0.989211,0.959750,0.824373
1,S2_scores_disagreement,9,0.987503,0.953947,0.810392


In [18]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
)

state_results = []

for state_name, state in states.items():

    print(f"\nTraining diagnostic model for {state_name}...")

    clf = ExtraTreesClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    )

    clf.fit(state, y_val)

    probabilities = clf.predict_proba(state)[:, 1]
    predictions = (probabilities >= 0.5).astype(np.int8)

    state_results.append({
        "state": state_name,
        "dimensions": state.shape[1],
        "roc_auc": roc_auc_score(
            y_val,
            probabilities,
        ),
        "pr_auc": average_precision_score(
            y_val,
            probabilities,
        ),
        "f1": f1_score(
            y_val,
            predictions,
        ),
    })

state_comparison = (
    pd.DataFrame(state_results)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

state_comparison


Training diagnostic model for S1_scores_only...

Training diagnostic model for S2_scores_disagreement...

Training diagnostic model for S3_scores_disagreement_traffic...


,state,dimensions,roc_auc,pr_auc,f1
0,S3_scores_disagreement_traffic,14,0.992354,0.969068,0.879346
1,S1_scores_only,4,0.989211,0.959750,0.824373
2,S2_scores_disagreement,9,0.987503,0.953947,0.810392


In [19]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "evaluation_results"
STATE_DIR = RESULTS_DIR / "rl_states"

print("Project root:", PROJECT_ROOT)

Project root: /home/kalpe/projects/adaptive_rl_anomaly_detection


In [20]:
state_s3 = np.load(
    STATE_DIR / "state_s3.npy"
)

metadata = joblib.load(
    STATE_DIR / "state_metadata.joblib"
)

print("S3 shape:", state_s3.shape)
print("S3 features:")
for feature in metadata["S3"]:
    print(" -", feature)

S3 shape: (201664, 14)
S3 features:
 - if_score
 - lof_score
 - svm_score
 - ae_score
 - score_mean
 - score_std
 - score_min
 - score_max
 - score_range
 - Average Packet Size
 - Packet Length Variance
 - Packet Length Std
 - Packet Length Mean
 - Avg Bwd Segment Size


In [22]:
PROCESSED_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "datasets"
    / "processed"
    / "cicids2017_processed.parquet"
)

df = pd.read_parquet(PROCESSED_PATH)

X = df.drop(columns=["Label"])
y = (df["Label"] != 0).astype(int)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_dev, X_val, y_dev, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.10,
    random_state=42,
    stratify=y_train,
)

y_val = y_val.reset_index(drop=True)

print(y_val.shape)
print(y_val.value_counts().sort_index())

(201664,)
Label
0    167605
1     34059
Name: count, dtype: int64


In [23]:
model_scores = joblib.load(
    RESULTS_DIR / "model_scores.joblib"
)

print(model_scores.keys())

dict_keys(['if_scores', 'lof_scores', 'svm_scores', 'ae_scores'])


In [24]:
print(
    {
        k: np.asarray(v).shape
        for k, v in model_scores.items()
    }
)

{'if_scores': (504160,), 'lof_scores': (504160,), 'svm_scores': (504160,), 'ae_scores': (504160,)}


In [25]:
from src.models.isolation_forest import IsolationForestModel
from src.models.local_outlier_factor import LocalOutlierFactorModel
from src.models.one_class_svm import OneClassSVMModel
from src.models.autoencoder import AutoEncoderModel

MODEL_DIR = Path.cwd() / "trained_models"

if_model = IsolationForestModel.load(
    MODEL_DIR / "isolation_forest.joblib"
)

lof_model = LocalOutlierFactorModel.load(
    MODEL_DIR / "local_outlier_factor.joblib"
)

svm_model = OneClassSVMModel.load(
    MODEL_DIR / "one_class_svm.joblib"
)

ae_model = AutoEncoderModel.load(
    MODEL_DIR / "autoencoder.joblib"
)

raw_validation_scores = {
    "if": np.asarray(if_model.anomaly_score(X_val)),
    "lof": np.asarray(lof_model.anomaly_score(X_val)),
    "svm": np.asarray(svm_model.anomaly_score(X_val)),
    "ae": np.asarray(ae_model.anomaly_score(X_val)),
}

print({
    k: v.shape
    for k, v in raw_validation_scores.items()
})

2026-08-12 14:39:50 | INFO     | src.models.isolation_forest | Isolation Forest loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/isolation_forest.joblib
2026-08-12 14:39:50 | INFO     | src.models.local_outlier_factor | Local Outlier Factor loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/local_outlier_factor.joblib
2026-08-12 14:39:50 | INFO     | src.models.one_class_svm | One-Class SVM loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/one_class_svm.joblib
2026-08-12 14:39:50 | INFO     | src.models.autoencoder | AutoEncoder loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/autoencoder.joblib


{'if': (201664,), 'lof': (201664,), 'svm': (201664,), 'ae': (201664,)}


In [26]:
from src.ensemble import ScoreCombiner

combiner = ScoreCombiner(
    weights={
        "if": 0.25,
        "lof": 0.25,
        "svm": 0.25,
        "ae": 0.25,
    },
    normalization="percentile",
)

combiner.fit(raw_validation_scores)

validation_scores = (
    combiner.get_normalized_scores(
        raw_validation_scores
    )
)

print("✓ Validation scores normalized")

✓ Validation scores normalized


In [27]:
action_results = []

for action, model_name in enumerate(
    ["if", "lof", "svm", "ae"]
):

    scores = validation_scores[model_name]

    # Threshold using NORMAL validation traffic
    threshold = np.percentile(
        scores[y_val.to_numpy() == 0],
        95,
    )

    predictions = (
        scores >= threshold
    ).astype(np.int8)

    action_results.append({
        "action": action,
        "selected_model": model_name,
        "threshold": threshold,
        "precision": precision_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_val,
            scores,
        ),
        "pr_auc": average_precision_score(
            y_val,
            scores,
        ),
    })

a1_results = pd.DataFrame(action_results)

a1_results.sort_values(
    "f1",
    ascending=False,
)

,action,selected_model,threshold,precision,recall,f1,roc_auc,pr_auc
3,3,ae,0.861417,0.700122,0.574503,0.631122,0.899944,0.734257
0,0,if,0.929449,0.410950,0.171673,0.242177,0.749830,0.339557
2,2,svm,0.943911,0.259105,0.086057,0.129201,0.720339,0.303365
1,1,lof,0.953947,0.097653,0.026630,0.041848,0.458300,0.150721


In [28]:
weight_presets = {
    "equal": {
        "if": 0.25,
        "lof": 0.25,
        "svm": 0.25,
        "ae": 0.25,
    },

    "ae_heavy": {
        "if": 0.10,
        "lof": 0.05,
        "svm": 0.10,
        "ae": 0.75,
    },

    "balanced": {
        "if": 0.20,
        "lof": 0.10,
        "svm": 0.20,
        "ae": 0.50,
    },

    "recall_static": {
        "if": 0.1272,
        "lof": 0.0377,
        "svm": 0.0991,
        "ae": 0.7359,
    },

    "if_heavy": {
        "if": 0.60,
        "lof": 0.10,
        "svm": 0.20,
        "ae": 0.10,
    },
}

for name, weights in weight_presets.items():
    print(
        name,
        "sum =",
        sum(weights.values())
    )

equal sum = 1.0
ae_heavy sum = 1.0
balanced sum = 1.0
recall_static sum = 0.9999
if_heavy sum = 1.0


In [29]:
a2_results = []

for action, (preset_name, weights) in enumerate(
    weight_presets.items()
):

    weighted_scores = sum(
        weights[name] * validation_scores[name]
        for name in ["if", "lof", "svm", "ae"]
    )

    threshold = np.percentile(
        weighted_scores[
            y_val.to_numpy() == 0
        ],
        95,
    )

    predictions = (
        weighted_scores >= threshold
    ).astype(np.int8)

    a2_results.append({
        "action": action,
        "preset": preset_name,
        "threshold": threshold,
        "precision": precision_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_val,
            weighted_scores,
        ),
        "pr_auc": average_precision_score(
            y_val,
            weighted_scores,
        ),
    })

a2_results = pd.DataFrame(a2_results)

a2_results.sort_values(
    "f1",
    ascending=False,
)

,action,preset,threshold,precision,recall,f1,roc_auc,pr_auc
3,3,recall_static,0.845048,0.686996,0.540092,0.604751,0.875612,0.657604
1,1,ae_heavy,0.841874,0.686972,0.540033,0.604705,0.877739,0.657524
2,2,balanced,0.831729,0.642098,0.441469,0.523210,0.841553,0.538720
0,0,equal,0.806599,0.512194,0.258375,0.343482,0.780383,0.404079
4,4,if_heavy,0.857911,0.509165,0.255263,0.340048,0.764761,0.387159


In [31]:
rng = np.random.default_rng(42)

A3_RESULTS = 500

a3_results = []

for action in range(A3_RESULTS):

    weights = rng.dirichlet(
        np.ones(4)
    )

    dynamic_scores = (
        weights[0] * validation_scores["if"]
        + weights[1] * validation_scores["lof"]
        + weights[2] * validation_scores["svm"]
        + weights[3] * validation_scores["ae"]
    )

    threshold = np.percentile(
        dynamic_scores[
            y_val.to_numpy() == 0
        ],
        95,
    )

    predictions = (
        dynamic_scores >= threshold
    ).astype(np.int8)

    a3_results.append({
        "action": action,
        "if_weight": weights[0],
        "lof_weight": weights[1],
        "svm_weight": weights[2],
        "ae_weight": weights[3],
        "precision": precision_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y_val,
            dynamic_scores,
        ),
        "pr_auc": average_precision_score(
            y_val,
            dynamic_scores,
        ),
    })

a3_results = pd.DataFrame(a3_results)

In [32]:
a3_results.sort_values(
    "f1",
    ascending=False,
).head(10)

,action,if_weight,lof_weight,svm_weight,ae_weight,precision,recall,f1,roc_auc,pr_auc
443,443,0.017359,0.000288,0.151524,0.830829,0.696164,0.563816,0.623039,0.888642,0.691399
193,193,0.079925,0.004705,0.064413,0.850956,0.692835,0.555037,0.616328,0.887991,0.701157
345,345,0.007920,0.049012,0.051073,0.891995,0.692497,0.554156,0.615651,0.892681,0.699525
24,24,0.097203,0.024151,0.160574,0.718072,0.689213,0.545700,0.609117,0.874682,0.650151
48,48,0.129002,0.025946,0.061713,0.783338,0.688300,0.543381,0.607315,0.880201,0.677525
162,162,0.006457,0.016528,0.282342,0.694673,0.688195,0.543116,0.607109,0.872926,0.626398
383,383,0.110023,0.040898,0.019317,0.829762,0.685858,0.537244,0.602522,0.884132,0.686874
473,473,0.138604,0.054819,0.077170,0.729407,0.683306,0.530932,0.597558,0.874012,0.650328
182,182,0.159170,0.041099,0.171198,0.628532,0.682104,0.527996,0.595237,0.862027,0.613273
186,186,0.053856,0.082583,0.125707,0.737854,0.681137,0.525647,0.593375,0.876618,0.634931


In [33]:
a3_results.sort_values(
    "pr_auc",
    ascending=False,
).head(10)

,action,if_weight,lof_weight,svm_weight,ae_weight,precision,recall,f1,roc_auc,pr_auc
193,193,0.079925,0.004705,0.064413,0.850956,0.692835,0.555037,0.616328,0.887991,0.701157
345,345,0.007920,0.049012,0.051073,0.891995,0.692497,0.554156,0.615651,0.892681,0.699525
443,443,0.017359,0.000288,0.151524,0.830829,0.696164,0.563816,0.623039,0.888642,0.691399
383,383,0.110023,0.040898,0.019317,0.829762,0.685858,0.537244,0.602522,0.884132,0.686874
48,48,0.129002,0.025946,0.061713,0.783338,0.688300,0.543381,0.607315,0.880201,0.677525
401,401,0.225339,0.030275,0.040023,0.704363,0.679368,0.521389,0.589986,0.868462,0.651400
473,473,0.138604,0.054819,0.077170,0.729407,0.683306,0.530932,0.597558,0.874012,0.650328
24,24,0.097203,0.024151,0.160574,0.718072,0.689213,0.545700,0.609117,0.874682,0.650151
280,280,0.234851,0.029621,0.046444,0.689084,0.678963,0.520420,0.589213,0.866543,0.646268
425,425,0.067501,0.113846,0.020190,0.798463,0.673115,0.506709,0.578177,0.878688,0.639615


In [34]:
best_a1 = (
    a1_results
    .sort_values("f1", ascending=False)
    .iloc[0]
)

best_a2 = (
    a2_results
    .sort_values("f1", ascending=False)
    .iloc[0]
)

best_a3 = (
    a3_results
    .sort_values("f1", ascending=False)
    .iloc[0]
)

action_comparison = pd.DataFrame([
    {
        "action_space": "A1_model_selection",
        "f1": best_a1["f1"],
        "roc_auc": best_a1["roc_auc"],
        "pr_auc": best_a1["pr_auc"],
    },
    {
        "action_space": "A2_weight_presets",
        "f1": best_a2["f1"],
        "roc_auc": best_a2["roc_auc"],
        "pr_auc": best_a2["pr_auc"],
    },
    {
        "action_space": "A3_dynamic_weights",
        "f1": best_a3["f1"],
        "roc_auc": best_a3["roc_auc"],
        "pr_auc": best_a3["pr_auc"],
    },
])

action_comparison.sort_values(
    "f1",
    ascending=False,
)

,action_space,f1,roc_auc,pr_auc
0,A1_model_selection,0.631122,0.899944,0.734257
2,A3_dynamic_weights,0.623039,0.888642,0.691399
1,A2_weight_presets,0.604751,0.875612,0.657604


In [35]:
from pathlib import Path
import joblib

RL_DIR = PROJECT_ROOT / "evaluation_results" / "rl_action"
RL_DIR.mkdir(parents=True, exist_ok=True)

# Save all 500 random dynamic-weight experiments
joblib.dump(
    a3_results,
    RL_DIR / "a3_dynamic_weights_results.joblib",
)

# Best A3 configuration according to F1
best_a3 = (
    a3_results
    .sort_values("f1", ascending=False)
    .iloc[0]
)

best_a3_config = {
    "if_weight": float(best_a3["if_weight"]),
    "lof_weight": float(best_a3["lof_weight"]),
    "svm_weight": float(best_a3["svm_weight"]),
    "ae_weight": float(best_a3["ae_weight"]),
    "f1": float(best_a3["f1"]),
    "roc_auc": float(best_a3["roc_auc"]),
    "pr_auc": float(best_a3["pr_auc"]),
}

joblib.dump(
    best_a3_config,
    RL_DIR / "a3_best_weights.joblib",
)

print("✓ A3 results saved")
print("Directory:", RL_DIR)
print("\nBest A3 configuration:")
print(best_a3_config)

✓ A3 results saved
Directory: /home/kalpe/projects/adaptive_rl_anomaly_detection/evaluation_results/rl_action

Best A3 configuration:
{'if_weight': 0.017358815603393297, 'lof_weight': 0.0002878540038612665, 'svm_weight': 0.15152402442278692, 'ae_weight': 0.8308293059699585, 'f1': 0.6230391123079668, 'roc_auc': 0.8886417667247393, 'pr_auc': 0.6913992515218956}


In [36]:
action_config = {
    "state": "S3",
    "action_type": "dynamic_weights",
    "action_dimension": 4,
    "models": ["if", "lof", "svm", "ae"],
    "weight_constraint": "weights_sum_to_1",
    "normalization": "percentile",
}

joblib.dump(
    action_config,
    RL_DIR / "action_config.joblib",
)

print("\n✓ Action configuration saved")


✓ Action configuration saved


In [37]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "evaluation_results"
RL_DIR = RESULTS_DIR / "rl_action"

print("Project root:", PROJECT_ROOT)

Project root: /home/kalpe/projects/adaptive_rl_anomaly_detection


In [38]:
STATE_DIR = RESULTS_DIR / "rl_states"

state_s3 = np.load(
    STATE_DIR / "state_s3.npy"
)

action_config = joblib.load(
    RL_DIR / "action_config.joblib"
)

best_a3 = joblib.load(
    RL_DIR / "a3_best_weights.joblib"
)

print("S3:", state_s3.shape)
print("Action config:", action_config)
print("Best A3 weights:", best_a3)

S3: (201664, 14)
Action config: {'state': 'S3', 'action_type': 'dynamic_weights', 'action_dimension': 4, 'models': ['if', 'lof', 'svm', 'ae'], 'weight_constraint': 'weights_sum_to_1', 'normalization': 'percentile'}
Best A3 weights: {'if_weight': 0.017358815603393297, 'lof_weight': 0.0002878540038612665, 'svm_weight': 0.15152402442278692, 'ae_weight': 0.8308293059699585, 'f1': 0.6230391123079668, 'roc_auc': 0.8886417667247393, 'pr_auc': 0.6913992515218956}


In [39]:
from sklearn.model_selection import train_test_split

PROCESSED_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "datasets"
    / "processed"
    / "cicids2017_processed.parquet"
)

df = pd.read_parquet(PROCESSED_PATH)

X = df.drop(columns=["Label"])
y = (df["Label"] != 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_dev, X_val, y_dev, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.10,
    random_state=42,
    stratify=y_train,
)

y_val = y_val.reset_index(drop=True)

print(y_val.shape)
print(y_val.value_counts().sort_index())

(201664,)
Label
0    167605
1     34059
Name: count, dtype: int64


In [40]:
REWARD_SCHEMES = {
    "R1_balanced": {
        "tp": 1.0,
        "tn": 1.0,
        "fp": -1.0,
        "fn": -1.0,
    },

    "R2_attack_sensitive": {
        "tp": 1.0,
        "tn": 1.0,
        "fp": -1.0,
        "fn": -2.0,
    },

    "R3_strong_attack_sensitive": {
        "tp": 1.0,
        "tn": 1.0,
        "fp": -1.0,
        "fn": -3.0,
    },
}

REWARD_SCHEMES

{'R1_balanced': {'tp': 1.0, 'tn': 1.0, 'fp': -1.0, 'fn': -1.0},
 'R2_attack_sensitive': {'tp': 1.0, 'tn': 1.0, 'fp': -1.0, 'fn': -2.0},
 'R3_strong_attack_sensitive': {'tp': 1.0, 'tn': 1.0, 'fp': -1.0, 'fn': -3.0}}

In [42]:
def calculate_rewards(
    y_true,
    y_pred,
    scheme,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rewards = np.zeros(len(y_true), dtype=np.float32)

    tp = (y_true == 1) & (y_pred == 1)
    tn = (y_true == 0) & (y_pred == 0)
    fp = (y_true == 0) & (y_pred == 1)
    fn = (y_true == 1) & (y_pred == 0)

    rewards[tp] = scheme["tp"]
    rewards[tn] = scheme["tn"]
    rewards[fp] = scheme["fp"]
    rewards[fn] = scheme["fn"]

    return rewards

In [43]:
from src.ensemble import ScoreCombiner

# We need the validation anomaly scores again.
from src.models.isolation_forest import IsolationForestModel
from src.models.local_outlier_factor import LocalOutlierFactorModel
from src.models.one_class_svm import OneClassSVMModel
from src.models.autoencoder import AutoEncoderModel

MODEL_DIR = Path.cwd() / "trained_models"

if_model = IsolationForestModel.load(
    MODEL_DIR / "isolation_forest.joblib"
)

lof_model = LocalOutlierFactorModel.load(
    MODEL_DIR / "local_outlier_factor.joblib"
)

svm_model = OneClassSVMModel.load(
    MODEL_DIR / "one_class_svm.joblib"
)

ae_model = AutoEncoderModel.load(
    MODEL_DIR / "autoencoder.joblib"
)

raw_scores = {
    "if": np.asarray(if_model.anomaly_score(X_val)),
    "lof": np.asarray(lof_model.anomaly_score(X_val)),
    "svm": np.asarray(svm_model.anomaly_score(X_val)),
    "ae": np.asarray(ae_model.anomaly_score(X_val)),
}

combiner = ScoreCombiner(
    weights={
        "if": 0.25,
        "lof": 0.25,
        "svm": 0.25,
        "ae": 0.25,
    },
    normalization="percentile",
)

combiner.fit(raw_scores)

normalized_scores = combiner.get_normalized_scores(
    raw_scores
)

print("✓ Validation scores ready")

2026-08-12 14:49:54 | INFO     | src.models.isolation_forest | Isolation Forest loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/isolation_forest.joblib
2026-08-12 14:49:54 | INFO     | src.models.local_outlier_factor | Local Outlier Factor loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/local_outlier_factor.joblib
2026-08-12 14:49:54 | INFO     | src.models.one_class_svm | One-Class SVM loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/one_class_svm.joblib
2026-08-12 14:49:54 | INFO     | src.models.autoencoder | AutoEncoder loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/autoencoder.joblib


✓ Validation scores ready


In [44]:
weights = np.array([
    best_a3["if_weight"],
    best_a3["lof_weight"],
    best_a3["svm_weight"],
    best_a3["ae_weight"],
])

dynamic_scores = (
    weights[0] * normalized_scores["if"]
    + weights[1] * normalized_scores["lof"]
    + weights[2] * normalized_scores["svm"]
    + weights[3] * normalized_scores["ae"]
)

print("Weights:", weights)
print("Weight sum:", weights.sum())
print("Score shape:", dynamic_scores.shape)

Weights: [1.73588156e-02 2.87854004e-04 1.51524024e-01 8.30829306e-01]
Weight sum: 1.0
Score shape: (201664,)


In [45]:
normal_mask = (
    y_val.to_numpy() == 0
)

reward_threshold = np.percentile(
    dynamic_scores[normal_mask],
    95,
)

dynamic_predictions = (
    dynamic_scores >= reward_threshold
).astype(np.int8)

print("Threshold:", reward_threshold)

print(
    "Prediction distribution:",
    np.unique(
        dynamic_predictions,
        return_counts=True,
    )
)

Threshold: 0.855413935312126
Prediction distribution: (array([0, 1], dtype=int8), array([174080,  27584]))


In [46]:
reward_results = []

for reward_name, scheme in REWARD_SCHEMES.items():

    rewards = calculate_rewards(
        y_true=y_val,
        y_pred=dynamic_predictions,
        scheme=scheme,
    )

    reward_results.append({
        "reward_scheme": reward_name,
        "mean_reward": rewards.mean(),
        "total_reward": rewards.sum(),
        "min_reward": rewards.min(),
        "max_reward": rewards.max(),
        "positive_rewards": np.sum(rewards > 0),
        "negative_rewards": np.sum(rewards < 0),
    })

reward_df = pd.DataFrame(reward_results)

reward_df

,reward_scheme,mean_reward,total_reward,min_reward,max_reward,positive_rewards,negative_rewards
0,R1_balanced,0.769547,155190.0,-1.0,1.0,178427,23237
1,R2_attack_sensitive,0.695880,140334.0,-2.0,1.0,178427,23237
2,R3_strong_attack_sensitive,0.622213,125478.0,-3.0,1.0,178427,23237


In [47]:
def reward_breakdown(
    y_true,
    y_pred,
    scheme,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = np.sum(
        (y_true == 1) & (y_pred == 1)
    )

    tn = np.sum(
        (y_true == 0) & (y_pred == 0)
    )

    fp = np.sum(
        (y_true == 0) & (y_pred == 1)
    )

    fn = np.sum(
        (y_true == 1) & (y_pred == 0)
    )

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP_reward": tp * scheme["tp"],
        "TN_reward": tn * scheme["tn"],
        "FP_penalty": fp * scheme["fp"],
        "FN_penalty": fn * scheme["fn"],
    }

In [48]:
for name, scheme in REWARD_SCHEMES.items():

    print("=" * 60)
    print(name)

    breakdown = reward_breakdown(
        y_val,
        dynamic_predictions,
        scheme,
    )

    for key, value in breakdown.items():
        print(f"{key:15}: {value}")

R1_balanced
TP             : 19203
TN             : 159224
FP             : 8381
FN             : 14856
TP_reward      : 19203.0
TN_reward      : 159224.0
FP_penalty     : -8381.0
FN_penalty     : -14856.0
R2_attack_sensitive
TP             : 19203
TN             : 159224
FP             : 8381
FN             : 14856
TP_reward      : 19203.0
TN_reward      : 159224.0
FP_penalty     : -8381.0
FN_penalty     : -29712.0
R3_strong_attack_sensitive
TP             : 19203
TN             : 159224
FP             : 8381
FN             : 14856
TP_reward      : 19203.0
TN_reward      : 159224.0
FP_penalty     : -8381.0
FN_penalty     : -44568.0


In [49]:
reward_df.sort_values(
    "mean_reward",
    ascending=False,
)

,reward_scheme,mean_reward,total_reward,min_reward,max_reward,positive_rewards,negative_rewards
0,R1_balanced,0.769547,155190.0,-1.0,1.0,178427,23237
1,R2_attack_sensitive,0.695880,140334.0,-2.0,1.0,178427,23237
2,R3_strong_attack_sensitive,0.622213,125478.0,-3.0,1.0,178427,23237


In [50]:
REWARD_DIR = (
    RESULTS_DIR
    / "rl_reward"
)

REWARD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

joblib.dump(
    reward_df,
    REWARD_DIR / "reward_comparison.joblib",
)

joblib.dump(
    REWARD_SCHEMES,
    REWARD_DIR / "reward_schemes.joblib",
)

print("✓ Reward experiment saved")
print("Directory:", REWARD_DIR)

✓ Reward experiment saved
Directory: /home/kalpe/projects/adaptive_rl_anomaly_detection/evaluation_results/rl_reward


In [51]:
FINAL_REWARD_CONFIG = {
    "name": "R1_balanced",
    "tp": 1.0,
    "tn": 1.0,
    "fp": -1.0,
    "fn": -1.0,
}

joblib.dump(
    FINAL_REWARD_CONFIG,
    REWARD_DIR / "final_reward_config.joblib",
)

print("✓ Final reward configuration saved")
print(FINAL_REWARD_CONFIG)

✓ Final reward configuration saved
{'name': 'R1_balanced', 'tp': 1.0, 'tn': 1.0, 'fp': -1.0, 'fn': -1.0}
